# Dimensionality Reduction & Clustering

Package-backed computational workflow for fitted gene-expression trajectories and dynamic-mode clustering. Publication plotting is intentionally kept out of this notebook for now.


In [1]:
from pathlib import Path
import json
import pandas as pd
import chronopack as cp


In [2]:
def read_original_notebook_dir(experiment_dir):
    text = (Path(experiment_dir) / "OriginalDirectoryPath.txt").read_text().strip()
    return Path(text.replace("~", str(Path.home()))).resolve()

workspace_dir = Path.cwd().resolve()
if workspace_dir.name == "k562-tnfa-puromycin.hg38":
    repo_root = workspace_dir.parent
else:
    repo_root = workspace_dir

puro_dir = repo_root / "k562-tnfa-puromycin.hg38"
k562a_dir = repo_root / "k562-tnfa.hg38.rep"
lps_dir = repo_root / "thp1-k562-lps.hg38.rep"

root_puro = read_original_notebook_dir(puro_dir).parent
root_k562a = read_original_notebook_dir(k562a_dir).parent
root_lps = read_original_notebook_dir(lps_dir).parent

SELF_CONTAINED_OUTDIR = puro_dir / "self_contained_outputs" / "global_analysis_extended"
SELF_CONTAINED_OUTDIR.mkdir(parents=True, exist_ok=True)

{
    "root_puro": str(root_puro),
    "root_k562a": str(root_k562a),
    "root_lps": str(root_lps),
    "output": str(SELF_CONTAINED_OUTDIR),
}


{'root_puro': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/manuscript_full_pipeline_20260919_165539/k562-tnfa-puromycin.hg38', 'root_k562a': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/manuscript_full_pipeline_20260919_165539/k562-tnfa.hg38.rep', 'root_lps': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/manuscript_full_pipeline_20260919_165539/thp1-k562-lps.hg38.rep', 'output': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/manuscript_full_pipeline_20260919_165539/k562-tnfa-puromycin.hg38/self_contained_outputs/global_analysis_extended'}

## Characterize Fitted Gene Trajectories

This extracts fitted curves from fixed tradeSeq sidecars, aligns/rescales time grids, standardizes fitted trajectories, computes derivatives, and calculates trajectory descriptors.


In [3]:
char = cp.tl.characterize_gene_trajectories(
    root_puro=root_puro,
    root_k562a=root_k562a,
    root_lps=root_lps,
    output_dir=SELF_CONTAINED_OUTDIR,
    time_map={"K562a": 110, "K562_puro": 110, "K562b": 165, "THP1": 165},
    offset_map={"K562a": 0, "K562_puro": 0, "K562b": -68, "THP1": -10},
    scale_map={"THP1": 0.65},
)

char.adata.write_h5ad(SELF_CONTAINED_OUTDIR / "gene_trajectory_characterization.h5ad")
{
    "adata_shape": char.adata.shape,
    "time_grid": char.metadata["time_grid"],
    "pc_df_characterized_rows": None if char.pc_df is None else char.pc_df.shape[0],
}


[K562a] TMAX=110, scale=1, offset=0 => aligned range 0.00..110.00, resampled to 110 minute points
[K562_puro] TMAX=110, scale=1, offset=0 => aligned range 0.00..110.00, resampled to 110 minute points
[K562b] TMAX=165, scale=1, offset=-68 => aligned range 0.00..97.00, resampled to 97 minute points
[THP1] TMAX=165, scale=0.65, offset=-10 => aligned range 0.40..97.25, resampled to 97 minute points
Shortest aligned end time across keys after rescaling: 97 min
Final shared aligned minute range: 1 .. 97  (n = 97 )


R callback write-console: Loading fixed tradeSeq objects: K562_puro
  
R callback write-console: Loading fixed tradeSeq objects: K562a
  
R callback write-console: Loading fixed tradeSeq objects: K562b
  
R callback write-console: Loading fixed tradeSeq objects: THP1
  
R callback write-console: Loading required package: SingleCellExperiment
  
R callback write-console: Loading required package: SummarizedExperiment
  
R callback write-console: Loading required package: MatrixGenerics
  
R callback write-console: Loading required package: matrixStats
  
R callback write-console: 
Attaching package: 'MatrixGenerics'

  
R callback write-console: The following objects are masked from 'package:matrixStats':

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantil

{'adata_shape': (0, 6723), 'time_grid': [0.0103092783505155, 0.0206185567010309, 0.0309278350515464, 0.0412371134020619, 0.0515463917525773, 0.0618556701030928, 0.0721649484536082, 0.0824742268041237, 0.0927835051546392, 0.103092783505155, 0.11340206185567, 0.123711340206186, 0.134020618556701, 0.144329896907216, 0.154639175257732, 0.164948453608247, 0.175257731958763, 0.185567010309278, 0.195876288659794, 0.206185567010309, 0.216494845360825, 0.22680412371134, 0.237113402061856, 0.247422680412371, 0.257731958762887, 0.268041237113402, 0.278350515463918, 0.288659793814433, 0.298969072164948, 0.309278350515464, 0.319587628865979, 0.329896907216495, 0.34020618556701, 0.350515463917526, 0.360824742268041, 0.371134020618557, 0.381443298969072, 0.391752577319588, 0.402061855670103, 0.412371134020619, 0.422680412371134, 0.43298969072165, 0.443298969072165, 0.45360824742268, 0.463917525773196, 0.474226804123711, 0.484536082474227, 0.494845360824742, 0.505154639175258, 0.515463917525773, 0.525

## Cluster Gene Trajectories

This preserves the current Ward.D2 signed clustering workflow with alpha-weighted trajectory and derivative distances. Native tradeSeq sidecars are retained upstream for plotting compatibility.


In [4]:
cluster = cp.tl.cluster_gene_trajectories(
    adata=char.adata,
    characterization=char,
    output_dir=SELF_CONTAINED_OUTDIR,
)

cluster.adata.write_h5ad(SELF_CONTAINED_OUTDIR / "gene_trajectory_clusters.h5ad")
pc_df = cluster.pc_df
pc_df.to_csv(SELF_CONTAINED_OUTDIR / "pc_df.new.csv")

{
    "adata_shape": cluster.adata.shape,
    "pc_df_shape": pc_df.shape,
    "broad_cluster_counts": pc_df["broad_cluster"].astype(str).value_counts().to_dict(),
}


{'adata_shape': (0, 6723), 'pc_df_shape': (6723, 35), 'broad_cluster_counts': {'trend-up': 1815, 'trend-down_1': 1561, 'trend-down_2': 1382, 'impulse-up_2': 657, 'transient-down': 568, '4': 391, 'impulse-up_1': 349}}

## Inspect Outputs


In [5]:
display(pc_df.head())
print(pc_df.shape)
print(pc_df["broad_cluster"].value_counts())


(6723, 35)
broad_cluster
trend-up          1815
trend-down_1      1561
trend-down_2      1382
impulse-up_2       657
transient-down     568
4                  391
impulse-up_1       349
Name: count, dtype: int64


,gene,cell_type,t_onset,t_peak,baseline_exp,peak_exp,peak_dlogexp,peak_dexp,peak_log2fc,auc_norm,owp,owp_ddt,mean_curv,max_curv,mean_turn,max_turn,waldStat_1,meanLogFC,pvalue_1,qval_1,PC1,PC2,PC3,PC4,cluster_broad_induced,cluster_dist_broad_induced,cluster_fine_induced,cluster_dist_fine_induced,cluster_broad_repressed,cluster_dist_broad_repressed,cluster_fine_repressed,cluster_dist_fine_repressed,broad_cluster,cluster_dist_broad,cluster_dist_fine
NFKBIA_K562a,NFKBIA,K562a,13.0,44.0,0.211178,1.669852,1.458673,4.076248,4.196603,0.658952,0.872347,0.253980,2.959513,17.237707,0.012753,0.058442,1967.377559,2.806293,0.0,0.0,-6.756087,-7.889632,1.875974,-1.647195,impulse-up_1,0.619030,1,0.433051,-1,NaN,-1,NaN,impulse-up_1,0.619030,0.433051
CXCL3_K562a,CXCL3,K562a,12.0,40.0,0.075631,0.956644,0.881013,1.524381,4.350700,0.714790,0.922803,0.245611,3.906862,27.670791,0.016150,0.092650,742.323899,2.786376,0.0,0.0,-8.008214,-6.514470,2.484261,-0.723473,impulse-up_1,0.785628,1,0.618291,-1,NaN,-1,NaN,impulse-up_1,0.785628,0.618291
CD69_K562a,CD69,K562a,14.0,41.0,0.031878,0.850409,0.818530,1.308211,5.371098,0.610452,0.810230,0.228689,4.635147,32.339366,0.019522,0.107502,602.364734,3.454632,0.0,0.0,-6.098681,-7.917358,2.437351,-1.663588,impulse-up_1,0.644811,1,0.502130,-1,NaN,-1,NaN,impulse-up_1,0.644811,0.502130
CD83_K562a,CD83,K562a,21.0,97.0,0.025841,0.361148,0.335307,0.408798,4.054501,0.423944,1.000000,0.865292,0.886568,1.690438,0.003968,0.007025,566.998711,2.128998,0.0,0.0,-10.974100,-0.391252,-0.789571,-0.194559,trend-up,0.477085,2,0.406509,-1,NaN,-1,NaN,trend-up,0.477085,0.406509
ARL4A_K562a,ARL4A,K562a,16.0,91.0,1.103865,1.487369,0.383503,1.409635,0.764934,0.556079,0.999390,0.647099,2.276590,10.048096,0.008902,0.033830,466.569370,0.365352,0.0,0.0,-10.763495,-2.424813,-0.557401,-0.319144,impulse-up_2,0.405166,3,0.320099,-1,NaN,-1,NaN,impulse-up_2,0.405166,0.320099


## Optional Reference Validation


In [6]:
import os

if os.environ.get("CHRONOPACK_VALIDATE_REFERENCE") == "1":
    reference_path = root_puro / "intmd" / "global_analysis" / "extended" / "pc_df.new.rds"
    print("Reference validation requested.")
    print("Reference path:", reference_path)
    print("Current rows:", pc_df.shape[0])
    print("Current columns:", pc_df.shape[1])
else:
    print("Set CHRONOPACK_VALIDATE_REFERENCE=1 to run external reference validation.")


Set CHRONOPACK_VALIDATE_REFERENCE=1 to run external reference validation.
